MVP


In [1]:
from datetime import datetime

import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import pandas as pd

In [2]:
data = fetch_california_housing()

In [3]:
X = data.data
y = data.target
feature_names = data.feature_names

In [4]:
feature_names

['MedInc',
 'HouseAge',
 'AveRooms',
 'AveBedrms',
 'Population',
 'AveOccup',
 'Latitude',
 'Longitude']

In [14]:
df = pd.DataFrame(data=X, columns=feature_names)
df.describe()


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [7]:
train_pool = Pool(X_train, y_train, feature_names=feature_names)
test_pool = Pool(X_test, y_test, feature_names=feature_names)

In [8]:
model = CatBoostRegressor(iterations=100, depth=6, learning_rate=0.1)

In [9]:
model.fit(train_pool, eval_set=test_pool, verbose=False)

CatBoostRegressor(depth=6, iterations=100, learning_rate=0.1, loss_function='RMSE')

In [10]:
predictions = model.predict(test_pool)

mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print(f"RMSE: {rmse:.4f}")
print(f"R2 Score: {r2:.4f}")
print("\nВажность признаков:")
for name, importance in zip(feature_names, model.feature_importances_):
    print(f"{name}: {importance:.2f}")

RMSE: 0.5151
R2 Score: 0.7995

Важность признаков:
MedInc: 37.39
HouseAge: 4.52
AveRooms: 2.40
AveBedrms: 0.92
Population: 0.70
AveOccup: 16.27
Latitude: 19.87
Longitude: 17.94


In [11]:
version = datetime.now().strftime("%Y%m%d_%H%M%S")
model.save_model(f"../artifacts/model_v{version}.cbm")